# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Get the metadata as a dictionary
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}\n")
print(f"Published: {metadata['datePublished']}")
print(f"Cite as: {metadata['citeAs']}")
if 'keywords' in metadata:
    print(f"Keywords: {', '.join(metadata['keywords'])}")
print(f"RecordSets: {len(metadata.get('recordSet', []))}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list the record sets, their `@id`s, and associated fields, referencing everything by their `@id` values.

In [ ]:
from pprint import pprint

# List all record sets and their fields
metadata = dataset.metadata.to_json()  # refresh to make sure we have the latest version

record_sets = metadata.get('recordSet', [])
if not record_sets:
    print('No record sets found in the schema. Please inspect the package or the documentation.')
else:
    for recset in record_sets:
        rs_id = recset['@id'] if isinstance(recset, dict) and '@id' in recset else recset
        # Try to fetch full record set definition (sometimes only @id are listed)
        full_recset = None
        # Top-level record sets might be expanded in 'recordSet', or just a list of @id
        if isinstance(recset, dict) and 'field' in recset:
            full_recset = recset
        else:
            # Look in '@graph' if available
            for obj in metadata.get('@graph', []):
                if obj.get('@id') == rs_id:
                    full_recset = obj
                    break
        print(f"RecordSet @id: {rs_id}")
        if full_recset and 'field' in full_recset:
            fields = full_recset['field']
            if isinstance(fields, dict):
                fields = [fields]
            print("  Fields and @id values:")
            for field in fields:
                fid = field['@id'] if isinstance(field, dict) and '@id' in field else field
                print(f"    - {fid}")
        else:
            print("  Fields: (Not expanded in metadata, use dataset API for details)")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** Here, we will attempt to auto-discover record set IDs in the dataset, if present. If not, refer to the Croissant schema or mlcroissant's `dataset.record_set_ids` property.

In [ ]:
record_set_ids = []

try:
    record_set_ids = dataset.record_set_ids
except AttributeError:
    # Fallback: try to read from the metadata
    metadata = dataset.metadata.to_json()
    for recset in metadata.get('recordSet', []):
        if isinstance(recset, dict) and '@id' in recset:
            record_set_ids.append(recset['@id'])
        elif isinstance(recset, str):
            record_set_ids.append(recset)

# If still nothing found, the dataset does not have standard record sets
if not record_set_ids:
    raise ValueError("No record sets found. This dataset might not be structured as a tabular record set.")

# Display the discovered record sets
print('Discovered record sets:')
for ridx, rsid in enumerate(record_set_ids):
    print(f"  {ridx}: {rsid}")

# Extract data from each record set into a DataFrame dictionary
dataframes = {}
for rec_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {rec_id}")
    except Exception as e:
        print(f"Error loading record set {rec_id}: {e}")

# Show a sample of one main record set (here, we pick the first one)
main_record_set_id = record_set_ids[0]
print(f"\nColumns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like outlier filtering, normalization, and grouping.

**You may need to update the `numeric_field_id` and `group_field_id` variables with the correct field `@id` from the printed column list above.**

In [ ]:
# Select record set for EDA
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Replace these with actual @id names after viewing df.columns
numeric_field_id = None
possible_numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
if possible_numeric_columns:
    numeric_field_id = possible_numeric_columns[0]
else:
    print("No numeric columns found to use for filtering and normalization.")

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].count() > 0 else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (mean): {filtered_df.shape[0]} records")

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records (head):")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by another field if available
    group_field_id = None
    # Look for a likely categorical column that's not the chosen numeric field
    candidate_groups = [c for c in df.columns if c != numeric_field_id and df[c].dtype == 'object']
    if candidate_groups:
        group_field_id = candidate_groups[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(f"\nGrouped by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA. Please check the columns and select a valid field.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we provide an example histogram and a boxplot for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # Boxplot by group field
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field identified for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**In this notebook, we loaded metadata and explored record sets and fields using `mlcroissant` for the dataset 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells'. After loading data into DataFrames, we performed basic EDA with outlier filtering, normalization, grouping, and visualization for the available numeric field(s). For deeper insights, be sure to inspect the schema and field definitions for scientific or domain-specific analysis.**